# 🧠 MS Lesion Segmentation: Clinical AI Dashboard
### Stage 2: Live Inference & Longitudinal Progression Demo

This notebook provides a professional, interactive interface for demonstrating automated MS lesion detection and tracking.

## 🛠 2.1 Minimal Environment Setup
Installing UI and medical imaging libraries.

In [ ]:
!pip install -q monai[all] SimpleITK nibabel gradio matplotlib

import os, torch, SimpleITK as sitk, numpy as np
import matplotlib.pyplot as plt
from monai.inferers import sliding_window_inference
from scipy.ndimage import label
import gradio as gr

# Environment detection — identical pattern to training notebook
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ENVIRONMENT = "colab"
    MODEL_PATH  = "/content/drive/MyDrive/MS_Project/models/best_model.pth"
except ImportError:
    ENVIRONMENT = "local"
    MODEL_PATH  = "./models/best_model.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE_NAME = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
print(f"Environment : {ENVIRONMENT}")
print(f"Device      : {device} ({DEVICE_NAME})")
print(f"Model path  : {MODEL_PATH}")
print("Dashboard Environment Ready.")

## 🧠 2.2 The Clinical Engine
This cell contains the 'One-Click' logic: Preprocessing, Inference, and Visual Overlays.

In [ ]:
import utils
from utils import (
    get_model, apply_skull_strip, preprocess_for_demo,
    create_mosaic, create_change_mosaic,
    run_lesion_stats, remove_fp_noise, segment_volume,
    compute_confidence_metrics, compute_accuracy_metrics,
    run_ai_analysis, run_longitudinal_analysis,
    PATCH_SIZE, OVERLAY_DIR,
)
print("Clinical Engine loaded from utils.py")

## 🚀 2.3 Interactive Dashboard (Gradio)
Run this cell to launch your presentation UI.

In [ ]:
model = get_model().to(device)
if os.path.exists(MODEL_PATH):
    ckpt = torch.load(MODEL_PATH, map_location=device)
    # Support both raw state-dict and full checkpoint dicts
    state = ckpt.get("model", ckpt) if isinstance(ckpt, dict) else ckpt
    model.load_state_dict(state)
    model.eval()
    print(f"Model loaded from {MODEL_PATH}")
else:
    print(f"WARNING: Model not found at {MODEL_PATH}. Run training first.")

# Make model available to utils functions (segment_volume etc.)
utils.model  = model
utils.device = device

css = """
body { font-family: 'Segoe UI', sans-serif; }
.gr-button-primary { background: #2563eb !important; }
"""

with gr.Blocks(title="MS Lesion AI", theme=gr.themes.Soft(), css=css) as demo:
    gr.Markdown("# MS Lesion Segmentation — AI Diagnostic Dashboard")
    gr.Markdown("3D BasicUNet (Ablation C) trained on MSLesSeg + Mendeley + Long-MR-MS datasets.")

    with gr.Tab("Single-Scan Analysis"):
        gr.Markdown(
            "Upload **FLAIR, T1, and T2** MRI scans (`.nii` or `.nii.gz`) to detect MS lesions.  "
            "Optionally upload a ground-truth mask to see accuracy metrics."
        )
        with gr.Row():
            with gr.Column(scale=1):
                s_flair  = gr.File(label="FLAIR MRI (NIfTI)", file_types=[".nii", ".gz"])
                s_t1     = gr.File(label="T1 MRI (NIfTI)",    file_types=[".nii", ".gz"])
                s_t2     = gr.File(label="T2 MRI (NIfTI)",    file_types=[".nii", ".gz"])
                gt_input = gr.File(label="Ground Truth Mask — optional (.nii / .nii.gz)",
                                   file_types=[".nii", ".gz"])
                s_btn    = gr.Button("Analyze Lesions", variant="primary")
            with gr.Column(scale=2):
                s_img    = gr.Image(label="Axial | Coronal | Sagittal Overlay")
                s_report = gr.Markdown()
        s_btn.click(run_ai_analysis,
                    inputs=[s_flair, s_t1, s_t2, gt_input],
                    outputs=[s_img, s_report])

    with gr.Tab("Longitudinal Tracking"):
        gr.Markdown(
            "Upload **Baseline (T0)** and **Follow-up (T1)** scans — three modalities each — "
            "to detect new disease activity."
        )
        gr.Markdown("> Scans are rigidly registered before comparison — no manual alignment needed.")
        with gr.Row():
            with gr.Column():
                gr.Markdown("#### Baseline (T0)")
                t0_flair = gr.File(label="T0 FLAIR (.nii / .nii.gz)", file_types=[".nii", ".gz"])
                t0_t1    = gr.File(label="T0 T1 (.nii / .nii.gz)",    file_types=[".nii", ".gz"])
                t0_t2    = gr.File(label="T0 T2 (.nii / .nii.gz)",    file_types=[".nii", ".gz"])
            with gr.Column():
                gr.Markdown("#### Follow-up (T1)")
                t1_flair = gr.File(label="T1 FLAIR (.nii / .nii.gz)", file_types=[".nii", ".gz"])
                t1_t1    = gr.File(label="T1 T1 (.nii / .nii.gz)",    file_types=[".nii", ".gz"])
                t1_t2    = gr.File(label="T1 T2 (.nii / .nii.gz)",    file_types=[".nii", ".gz"])
        l_btn = gr.Button("Detect Progression", variant="primary")
        with gr.Row():
            l_img_t0  = gr.Image(label="Baseline Segmentation (T0)")
            l_img_chg = gr.Image(label="Change Map  [Green = New   Blue = Resolved]")
        l_report = gr.Markdown()
        l_btn.click(run_longitudinal_analysis,
                    inputs=[t0_flair, t0_t1, t0_t2, t1_flair, t1_t1, t1_t2],
                    outputs=[l_img_t0, l_img_chg, l_report])

demo.launch(share=True)